In [1]:
from transformers import BartTokenizer
from transformers import BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch
from Utils import TimexNorm_Utils
from Reader import obtain_combined_dataset

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
utils = TimexNorm_Utils(tokenizer)

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\harry\.cache\huggingface\hub\models--facebook--bart-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [2]:
tokenizer.add_special_tokens({"additional_special_tokens": ["<timex","type=DATE>","type=TIME>","type=DURATION>","type=SET>","</timex>", "<sep>"]})
model.resize_token_embeddings(len(tokenizer))

Embedding(50272, 1024)

In [3]:
datasets = obtain_combined_dataset(["TempEval3","wikiwars","tweets"], "normalised")

In [4]:
datasets = utils.tokenize_datasets(datasets)

Map:   0%|          | 0/739 [00:00<?, ? examples/s]

Map:   0%|          | 0/15843 [00:00<?, ? examples/s]

Map:   0%|          | 0/1822 [00:00<?, ? examples/s]

In [5]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results/TimeNormBart",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=1,
    predict_with_generate=True,
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=500,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
    data_collator=data_collator,
    compute_metrics=utils.compute_metrics,
)

In [ ]:
trainer.train()

d:\GeoTKG\venv\Lib\site-packages\transformers\optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 15843
  Num Epochs = 1
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 991
  Number of trainable parameters = 406298624


  0%|          | 0/991 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
trainer.save_model("./results/TimeNormBart")
tokenizer.save_pretrained("./results/TimeNormBart")

In [ ]:
trainer.evaluate(datasets["test"])